# Libreta 14 — revisión técnica y cierre de la regresión histórica MCR004
Versión 1.0. Fase 1, paso 4.

La libreta 13 ya generó los 20 recortes. **No la repitas.** Esta libreta lee la misma ejecución, verifica su integridad y permite resolver las filas pendientes. No calcula nuevos recortes, no ajusta parámetros ni consulta desenlaces.

El CSV contiene 17 propuestas de revisión técnica asistida y tres casos pendientes. La versión del método era conocida durante la asistencia; no es una lectura clínica ni un segundo lector independiente. El investigador confirma o corrige las propuestas.

Ejecuta de arriba abajo en **Colab 2026.07, Python 3.12, CPU**. Usa el mismo secreto `KNEE_DATA_ROOT` de la libreta 13. Conserva la carpeta de salida y las evidencias originales.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
from IPython.display import display, HTML, Image
import ipywidgets as widgets
import hashlib, html, json, os, subprocess, sys, tempfile
drive.mount('/content/drive')
private_root_value = userdata.get('KNEE_DATA_ROOT')
if not private_root_value:
    raise RuntimeError('Falta el secreto KNEE_DATA_ROOT usado en la libreta 13.')
private_root_path = Path(private_root_value).resolve()
if not private_root_path.is_dir():
    raise FileNotFoundError('La raíz privada autorizada no está disponible.')
os.environ['KNEE_DATA_ROOT'] = str(private_root_path)
os.environ['KNEE_REVIEW_EXECUTION_ENVIRONMENT'] = 'Google Colab, libreta 14'

def run_checked(command, cwd=None):
    result = subprocess.run([str(x) for x in command], cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        print(result.stdout or '(sin salida estándar)')
        print(result.stderr or '(sin detalle de error)')
        raise RuntimeError(f'La operación se detuvo con código {result.returncode}. No se autoriza continuar.')
    return result.stdout

In [ ]:
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Selecciona Colab 2026.07 / Python 3.12 y vuelve a ejecutar desde el inicio.')
task_dir = Path(tempfile.mkdtemp(prefix='mcr004_revision_', dir='/content'))
REPO_DIR = task_dir/'Knee'
run_checked(['git', 'clone', '--depth', '1', '--branch',
             'codex/fase-0-base-reproducible', 'https://github.com/AlonsoYess/Knee.git', REPO_DIR])
closure_git_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR).strip()
config_path = REPO_DIR/'configs'/'roi_mcr004_historical.example.json'
config = json.loads(config_path.read_text(encoding='utf-8'))
output_dir = (private_root_path/config['output_dir']).resolve()
if not output_dir.is_relative_to(private_root_path) or not output_dir.is_dir():
    raise RuntimeError('La salida histórica existente no está disponible bajo la raíz autorizada.')
print('Código de revisión:', closure_git_commit)

## Entorno y comprobaciones
Se crea un entorno aislado mediante el arranque ya corregido de la libreta 13 (`virtualenv`, sin depender de `ensurepip`). Las pruebas son sintéticas y no miden el rendimiento anatómico. La creación del entorno no vuelve a procesar el piloto.

In [ ]:
VENV_DIR = task_dir/'venv_review'
run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv==21.7.11'])
run_checked([sys.executable, '-m', 'virtualenv', '--no-periodic-update', VENV_DIR])
venv_python = VENV_DIR/'bin'/'python'
run_checked([venv_python, '-m', 'pip', 'install', '-q', '-r', REPO_DIR/'requirements-mcr004.txt'])
run_checked([venv_python, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
run_checked([venv_python, '-m', 'unittest', 'discover', '-s', 'tests',
             '-p', 'test_roi_mcr004*.py', '-v'], cwd=REPO_DIR)
tests_passed = True
print('Pruebas técnicas del entorno: correctas.')

def review_call(action, *arguments):
    raw = run_checked([venv_python, '-m', 'knee.roi_mcr004_review',
                       '--config', config_path, '--action', action, *arguments], cwd=REPO_DIR)
    return json.loads(raw)

context = review_call('context')
print('Filas de revisión:', len(context['review']))
print('Casos pendientes:', ', '.join(context['pending_knee_aliases']))
print('No se ha ejecutado ningún cierre.')

## Propuestas registradas
La siguiente tabla muestra las veinte filas. Las propuestas no son resultados definitivos hasta tu confirmación. Si discrepas con alguna de las 17 propuestas, puedes seleccionarla en el visor y registrar la corrección; no necesitas modificar archivos manualmente.

In [ ]:
table = '<table><tr><th>Rodilla</th><th>Propuesta</th><th>Observación</th></tr>'
for row in context['review']:
    table += '<tr>' + ''.join('<td style="padding:6px;border:1px solid #ddd;vertical-align:top">' +
        html.escape(value) + '</td>' for value in (
            row['knee_alias'], row['decision'] or 'PENDIENTE', row['reviewer_notes'])) + '</tr>'
display(HTML(table+'</table>'))

## Resolver las tres filas pendientes
Selecciona cada caso. Izquierda: mitad original y caja cian; derecha: recorte existente. Ajusta la ventana si faltan detalles. **El contraste solo cambia la visualización, nunca los píxeles guardados ni la geometría.** El botón «Contraste de la ROI» propone límites de visualización, no una decisión.

Guía aprobada:
- Cobertura: ambos compartimentos completos, cóndilos distales, mesetas tibiales y hueso adyacente.
- Encuadre: región articular de ambos compartimentos entre el 25 % y el 75 % de la altura, con márgenes laterales óseos visibles. No exige una línea articular recta.
- Contaminación crítica: texto legible o rectángulo de anonimización en cualquier lugar del recorte; dispositivo/elemento externo sobre hueso o articulación; relleno artificial.
- Advertencia periférica: solo puntos aislados de regla o borde de dispositivo fuera del hueso y en el cuarto lateral externo.
- «No evaluable»: sigue sin poderse decidir después de ajustar la ventana. Deja vacíos los criterios desconocidos; no inventes «SI» o «NO».

En cada caso elige la decisión, comprueba los criterios, escribe una observación y marca que cotejaste original/ROI. «SI» en contaminación significa que está presente. Si no puedes identificar la naturaleza de una marca, documenta la duda; no necesitas emitir un diagnóstico.

In [ ]:
flag_labels = {
    'laterality_ok_yes_no': 'Lado correcto',
    'coverage_ok_yes_no': 'Cobertura completa',
    'frame_ok_yes_no': 'Encuadre correcto',
    'visualizable_yes_no': 'Se puede evaluar',
    'critical_contamination_yes_no': 'Contam. crítica presente',
    'peripheral_warning_yes_no': 'Advertencia periférica',
    'outcome_blinded_yes_no': 'Sin consultar desenlaces',
}
decision_options = [('PENDIENTE', ''), ('Aceptable', 'aceptable'),
    ('Aceptable con advertencia periférica', 'aceptable_con_advertencia_periferica'),
    ('Rechazado', 'rechazado'), ('No evaluable tras ajustar ventana', 'no_evaluable')]
forms = {}
viewer_output = widgets.Output()
selector = widgets.Dropdown(
    options=context['pending_knee_aliases'] + [r['knee_alias'] for r in context['review']
        if r['candidate_status']=='candidate' and r['knee_alias'] not in context['pending_knee_aliases']],
    description='Rodilla:', layout=widgets.Layout(width='440px'))

def create_form(alias):
    row = next(r for r in context['review'] if r['knee_alias']==alias)
    view = review_call('view', '--alias', alias)
    span = view['pixel_max']-view['pixel_min']
    slider = widgets.FloatRangeSlider(value=(view['display_low'],view['display_high']),
        min=view['pixel_min'], max=view['pixel_max'], step=max(span/2048,0.01),
        description='Ventana:', continuous_update=False, readout_format='.1f',
        layout=widgets.Layout(width='780px'))
    apply_button = widgets.Button(description='Aplicar ventana')
    roi_button = widgets.Button(description='Contraste de la ROI')
    image_output = widgets.Output()
    inspected = widgets.Checkbox(value=False, description='Cotejé original y ROI con ventana suficiente')
    include = widgets.Checkbox(value=alias in context['pending_knee_aliases'],
        description='Registrar esta decisión/corrección en el cierre')
    decision = widgets.Dropdown(options=decision_options, value=row['decision'],
        description='Decisión:', layout=widgets.Layout(width='680px'))
    flags = {key:widgets.Dropdown(options=[('Desconocido',''),('SI','SI'),('NO','NO')],
        value=row[key], description=label, style={'description_width':'210px'},
        layout=widgets.Layout(width='420px')) for key,label in flag_labels.items()}
    notes = widgets.Textarea(value='',description='Observación:',
        placeholder='Describe el motivo técnico. No escribas nombres ni identificadores personales.',
        layout=widgets.Layout(width='900px',height='100px'),style={'description_width':'110px'})
    state = {'view':view,'decision':decision,'flags':flags,'notes':notes,
             'inspected':inspected,'include':include}

    def show_image():
        with image_output:
            image_output.clear_output(wait=True)
            display(Image(filename=str(output_dir/state['view']['display_file']),width=1200))
            print('Ventana:', state['view']['display_low'], state['view']['display_high'])

    def apply_window(_):
        low,high = slider.value
        if high <= low:
            with image_output:
                print('El límite superior debe ser mayor que el inferior.')
            return
        try:
            state['view'] = review_call('view','--alias',alias,'--low',str(low),'--high',str(high))
            inspected.value = False
            show_image()
        except Exception as exc:
            with image_output:
                print(str(exc))

    def suggest_window(_):
        low,high = state['view']['suggested_roi_window']
        if high > low:
            slider.value = (low,high)
            apply_window(None)
        else:
            with image_output:
                print('La ROI no ofrece rango de contraste. Usa la ventana manual y documenta si no es evaluable.')

    apply_button.on_click(apply_window)
    roi_button.on_click(suggest_window)
    show_image()
    state['panel'] = widgets.VBox([
        widgets.HTML('<b>'+html.escape(alias)+'</b><p>'+html.escape(row['reviewer_notes'])+'</p>'),
        slider,widgets.HBox([apply_button,roi_button]),image_output,inspected,include,decision,
        *flags.values(),notes])
    return state

def select_case(change=None):
    alias = selector.value
    with viewer_output:
        viewer_output.clear_output(wait=True)
        try:
            if alias not in forms:
                forms[alias] = create_form(alias)
            display(forms[alias]['panel'])
        except Exception as exc:
            print(str(exc))

selector.observe(select_case,names='value')
display(selector,viewer_output)
select_case()

## Confirmación del investigador
Antes del cierre, revisa las tres rodillas pendientes usando el selector. Puedes volver a cada una sin perder lo que ingresaste. Las otras 17 propuestas se conservan si estás de acuerdo. Si registraste una corrección adicional, también se verificará.

Marca las dos confirmaciones solo si son ciertas. La celda siguiente se detendrá si falta una decisión, una observación, un cotejo o alguna confirmación. No cambia los umbrales para aprobar el método.

In [ ]:
confirm_all = widgets.Checkbox(value=False,
    description='Confirmo las 20 filas: propuestas asistidas conservadas y decisiones/correcciones ingresadas')
blind_confirm = widgets.Checkbox(value=False,
    description='No consulté desenlaces, predicciones ni resultados futuros durante esta revisión técnica')
display(confirm_all,blind_confirm)

## Guardar revisión y cerrar solo la regresión histórica
Ejecuta esta celda **después de completar el formulario**. No uses «Ejecutar todo» sin detenerte en los formularios. Se verifica nuevamente cada recorte nativo contra su radiografía autorizada y la separación congelada.

Se exige conjuntamente Q ≥ 19/20, F = 0 y W ≤ 2/20. Un candidato rechazado o no evaluable cuenta en F; no se convierte retrospectivamente en abstención. La revisión completa puede resultar rechazada. Incluso si supera la regresión, el paso 4 continúa abierto; esta libreta no selecciona una muestra nueva.

In [ ]:
if not tests_passed or not confirm_all.value or not blind_confirm.value:
    raise RuntimeError('Faltan las comprobaciones o tus dos confirmaciones; no se guardó el cierre.')
updates = {}
for alias,state in forms.items():
    if not state['include'].value:
        continue
    if not state['decision'].value or not state['inspected'].value or not state['notes'].value.strip():
        raise RuntimeError(f'Falta decisión, cotejo u observación en {alias}; no se guardó el cierre.')
    updates[alias] = {'decision':state['decision'].value,
        **{key:control.value for key,control in state['flags'].items()},
        'reviewer_notes':state['notes'].value.strip(),'window_reviewed':True,'view':state['view']}
missing = set(context['pending_knee_aliases'])-set(updates)
if missing:
    raise RuntimeError('Faltan casos pendientes: '+', '.join(sorted(missing))+'. No se guardó el cierre.')
response = {'draft_csv_sha256':context['review_csv_sha256'],
    'confirm_entire_assisted_review':confirm_all.value,'outcome_blinded':blind_confirm.value,
    'technical_tests_passed':tests_passed,
    'updates':updates}
response_path = task_dir/'respuestas_para_cierre.json'
response_path.write_text(json.dumps(response,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
closed_summary = review_call('close','--response',str(response_path),
                            '--git-commit',closure_git_commit)
print(json.dumps(closed_summary,ensure_ascii=False,indent=2))
print('Paso 4: abierto. No se ejecutaron etapas posteriores.')

## Resultado que debes compartir
Pega únicamente la salida final anterior. El CSV definitivo, la copia del borrador, las ventanas y el registro con tus respuestas quedan en la misma carpeta privada de Drive. No necesitas volver a enviar las veinte imágenes.

Si el cierre es rechazado, nos detenemos a documentar el resultado y a decidir contigo el siguiente cambio. Esta libreta no inicia otra prueba.